# CSC 485A Assignment 2: Hierarchical Parallelism

## Design summary

**DenseGraph::build_graph**: The matrix is zeroed with `cudaMemsetAsync`. One thread per edge, in a grid-stride loop, then sets `A[u*n + v] = 1`. The edge list already contains both directions, so the matrix stays symmetric.

**DenseGraph::two_hop_reachability**: This computes A² with *shared-memory tiling* (Lecture 09). Each 32×32 thread block loads one tile of A and one tile of B into `__shared__` memory, calls `__syncthreads()`, accumulates the partial dot products from shared memory, and moves on to the next tile. The result is clamped to {0,1} and the diagonal is cleared. `threadIdx.x` indexes columns, so global loads and stores are coalesced. The output goes to a second buffer, because squaring in place would race, and the wrapper then swaps the buffers.

**SparseGraph::build_graph**: The CSR is built with a *counting sort* on the edge sources (Counting Sort / Radix Sort notes):
1. histogram of degrees (`atomicAdd`)
2. device-wide exclusive scan to get `neighbours_start_at`
3. scatter, where each edge claims a slot with `atomicAdd` on a per-row cursor
4. a per-row sort, so the output is canonical (the atomic scatter is not stable)

The scan is hierarchical. Within a warp it is a Hillis-Steele scan using `__shfl_up_sync` (Lecture 15). Warp totals are combined through shared memory, and block totals are scanned recursively across kernel launches.

**SparseGraph::two_hop_reachability**: One thread block handles each source node `u`. Warps take different `v ∈ N(u)`, and the lanes read `N(v)` contiguously. Every `w ≠ u` is marked in a shared-memory bitmap with `atomicOr`, which removes duplicates for free. The bitmap is then compacted: `__popc` gives each thread its word's count, a block exclusive scan gives the write offset, and set bits are written in order, so the rows come out sorted. This runs twice. The first pass counts row sizes, which are scanned into new offsets. The second pass fills a newly allocated `neighbours` array, because the two-hop graph can have more than m edges. Work is O(Σ_v deg(v)² + n²/32), versus Θ(n³) for the dense GEMM, so CSR wins well below 2.5% density.

The benchmark at the end checks both implementations against a CPU reference and reports which one is faster at each density.

In [1]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-udzcgu69
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-udzcgu69
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.2-py3-none-any.whl size=12545 sha256=f293982cdc354d66d92473c4350316b6ff6623d7028219d8cf93377ffb402266
  Stored in directory: /tmp/pip-ephem-wheel-cache-8czbbqxn/wheels/2f/1e/79/584a8d85ad018ecc510baa90d4fbc161bd56a4d66943d5b0d4
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpx7nle6b_".


In [2]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [3]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [4]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [5]:
%%cuda_group_save -g "source" -n "scan.h"
/**
 * Hierarchical (warp -> block -> grid) exclusive prefix scan.
 *
 * Course sources: "Exposing DLP" (Hillis-Steele scan), "15: Warp Primitives"
 * (__shfl_up_sync), and the Counting Sort notes (histogram + exclusive scan).
 */
#pragma once

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {
namespace gpu     {

unsigned int constexpr full_mask  = 0xFFFFFFFFU;
int          constexpr warp_size  = 32;
int          constexpr scan_block = 1024; // threads per block for global scans

/**
 * Warp-level inclusive Hillis-Steele scan entirely in registers, using
 * __shfl_up_sync so lanes read each other's registers (no smem, no barrier).
 */
__device__ __forceinline__
int warp_inclusive_scan( int x )
{
    int const lane_id = threadIdx.x % warp_size;
    #pragma unroll
    for( int offset = 1; offset < warp_size; offset <<= 1 )
    {
        int const y = __shfl_up_sync( full_mask, x, offset );
        x += ( lane_id >= offset ) ? y : 0; // branch-free select
    }
    return x;
}

/**
 * Block-level exclusive scan: each warp scans in registers, warp totals
 * go through shared memory, and warp 0 scans those totals.
 *
 * @pre blockDim.x is a multiple of 32 and <= 1024; every thread in the block calls this.
 * @param warp_sums a __shared__ array of 32 ints
 * @param total     receives the sum over the whole block
 */
__device__ __forceinline__
int block_exclusive_scan( int x, int & total, int * warp_sums )
{
    int const lane_id   = threadIdx.x % warp_size;
    int const warp_id   = threadIdx.x / warp_size;
    int const num_warps = blockDim.x  / warp_size;

    int const inclusive = warp_inclusive_scan( x );
    if( lane_id == warp_size - 1 ) { warp_sums[ warp_id ] = inclusive; }
    __syncthreads();

    if( warp_id == 0 )
    {
        int const s = ( lane_id < num_warps ) ? warp_sums[ lane_id ] : 0;
        warp_sums[ lane_id ] = warp_inclusive_scan( s );
    }
    __syncthreads();

    int const warp_prefix = ( warp_id > 0 ) ? warp_sums[ warp_id - 1 ] : 0;
    total = warp_sums[ num_warps - 1 ];
    __syncthreads(); // warp_sums may be reused by the caller right after this

    return warp_prefix + inclusive - x;
}

/**
 * Scans each block's chunk of `in` independently and records each block's total.
 * Safe in place (in == out): each thread reads its element before any write.
 */
__global__
void scan_blocks( node_t const * in, node_t * out, std::size_t n, node_t * block_sums )
{
    __shared__ int warp_sums[ warp_size ];

    std::size_t const i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    int const x = ( i < n ) ? in[ i ] : 0;

    int total;
    int const prefix = block_exclusive_scan( x, total, warp_sums );

    if( i < n ) { out[ i ] = prefix; }
    if( threadIdx.x == 0 && block_sums != nullptr ) { block_sums[ blockIdx.x ] = total; }
}

/** Adds each block's scanned offset to every element in that block. */
__global__
void add_block_offsets( node_t * out, std::size_t n, node_t const * block_offsets )
{
    std::size_t const i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    if( i < n ) { out[ i ] += block_offsets[ blockIdx.x ]; }
}

} // namespace gpu

/**
 * Host-side device-wide exclusive scan: scan per block, recursively
 * scan the block totals, then add them back (grid-level synchronisation
 * comes from separate kernel launches).
 */
inline
void exclusive_scan( node_t const * d_in, node_t * d_out, std::size_t n )
{
    if( n == 0 ) { return; }

    std::size_t const num_blocks = ( n + gpu::scan_block - 1 ) / gpu::scan_block;
    if( num_blocks == 1 )
    {
        gpu::scan_blocks<<< 1, gpu::scan_block >>>( d_in, d_out, n, nullptr );
        return;
    }

    node_t * d_block_sums;
    CUDA_CALL( cudaMalloc( (void**)&d_block_sums, sizeof( node_t ) * num_blocks ) );

    gpu::scan_blocks<<< num_blocks, gpu::scan_block >>>( d_in, d_out, n, d_block_sums );
    exclusive_scan( d_block_sums, d_block_sums, num_blocks ); // in place, recursive
    gpu::add_block_offsets<<< num_blocks, gpu::scan_block >>>( d_out, n, d_block_sums );

    CUDA_CALL( cudaFree( d_block_sums ) );
}

} // namespace a2
} // namespace csc485b

In [6]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */
#pragma once

#include <algorithm> // std::min
#include <cstddef>   // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};


namespace gpu {

/** Tile width for shared-memory GEMM: one 32x32 = 1024-thread block per output tile. */
int constexpr dense_tile = 32;

/**
 * Constructs a DenseGraph from an input edge list of m edges.
 * One thread per edge (grid-stride loop), each setting one matrix cell.
 * The edge list already holds both (u,v) and (v,u), so the matrix stays symmetric.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 * @pre g.adjacencyMatrix has been zeroed (done by the host-side build_graph()).
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    std::size_t const stride = gridDim.x * static_cast< std::size_t >( blockDim.x );
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
         i < m; i += stride )
    {
        edge_t const e = edge_list[ i ];
        g.adjacencyMatrix[ static_cast< std::size_t >( e.x ) * g.n + e.y ] = 1;
    }
}

/**
  * Writes the two-hop neighbourhood of input graph g into `result`.
  *
  * Squares the adjacency matrix with shared-memory tiling (lecture 09):
  *   1. each thread loads one cell of a tile of A and one of B into smem,
  *   2. __syncthreads(),
  *   3. dot product over the tile from smem, __syncthreads(), next tile,
  * then removes the diagonal and clamps values back to [0,1].
  *
  * Out of place: squaring in place would race (other blocks still read
  * the cells being overwritten). The host-side wrapper swaps the buffers.
  *
  * threadIdx.x indexes columns, so each warp's global loads and stores are coalesced.
  */
__global__
void two_hop_reachability( DenseGraph g, node_t * result )
{
    __shared__ node_t a_tile[ dense_tile ][ dense_tile ];
    __shared__ node_t b_tile[ dense_tile ][ dense_tile ];

    std::size_t const n   = g.n;
    node_t const *    A   = g.adjacencyMatrix;
    int const         tx  = threadIdx.x;
    int const         ty  = threadIdx.y;
    std::size_t const row = blockIdx.y * static_cast< std::size_t >( dense_tile ) + ty;
    std::size_t const col = blockIdx.x * static_cast< std::size_t >( dense_tile ) + tx;

    int sum = 0;
    for( std::size_t t = 0; t < n; t += dense_tile )
    {
        // step 1: populate smem; out-of-range cells padded with 0
        std::size_t const a_col = t + tx;
        std::size_t const b_row = t + ty;
        a_tile[ ty ][ tx ] = ( row   < n && a_col < n ) ? A[ row   * n + a_col ] : 0;
        b_tile[ ty ][ tx ] = ( b_row < n && col   < n ) ? A[ b_row * n + col   ] : 0;
        __syncthreads();

        // step 2: compute with the cached tile (a_tile row is a broadcast,
        // b_tile row is conflict-free since consecutive tx -> consecutive banks)
        #pragma unroll
        for( int k = 0; k < dense_tile; ++k )
        {
            sum += a_tile[ ty ][ k ] * b_tile[ k ][ tx ];
        }
        __syncthreads(); // don't overwrite the tile while others still read it
    }

    // step 3: publish block-local result: clamp to {0,1} and drop self-loops
    if( row < n && col < n )
    {
        result[ row * n + col ] = ( row != col ) & ( sum > 0 );
    }
}

} // namespace gpu


/**
 * Host-side driver: zero the matrix, then scatter edges in one kernel.
 */
inline
void build_graph( DenseGraph & g, edge_t const * d_edges, std::size_t m )
{
    CUDA_CALL( cudaMemsetAsync( g.adjacencyMatrix, 0, sizeof( node_t ) * g.matrix_size() ) );

    int constexpr threads = 256;
    std::size_t const blocks = std::min< std::size_t >( ( m + threads - 1 ) / threads, 65535 );
    if( blocks > 0 ) { gpu::build_graph<<< blocks, threads >>>( g, d_edges, m ); }
    CHECK_ERROR( "dense build_graph" );
}

/**
 * Host-side driver: replaces g's adjacency matrix with its two-hop neighbourhood.
 */
inline
void two_hop_reachability( DenseGraph & g )
{
    node_t * d_result;
    CUDA_CALL( cudaMalloc( (void**)&d_result, sizeof( node_t ) * g.matrix_size() ) );

    unsigned const tiles = static_cast< unsigned >( ( g.n + gpu::dense_tile - 1 ) / gpu::dense_tile );
    dim3 const grid ( tiles, tiles );
    dim3 const block( gpu::dense_tile, gpu::dense_tile );
    gpu::two_hop_reachability<<< grid, block >>>( g, d_result );
    CHECK_ERROR( "dense two_hop_reachability" );

    // "in place" from the caller's perspective: swap in the new matrix
    CUDA_CALL( cudaFree( g.adjacencyMatrix ) );
    g.adjacencyMatrix = d_result;
}

} // namespace a2
} // namespace csc485b

In [7]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */
#pragma once

#include <algorithm> // std::min
#include <cstddef>   // std::size_t type
#include <cstdint>   // uint32_t

#include "cuda_common.h"
#include "data_types.h"
#include "scan.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */

  /** Start/end of node u's adjacency list (offset array has n entries, so the last row ends at m). */
  __device__ __forceinline__
  node_t row_begin( node_t u ) const { return neighbours_start_at[ u ]; }

  __device__ __forceinline__
  node_t row_end( node_t u ) const
  {
      return ( static_cast< std::size_t >( u ) + 1 < n ) ? neighbours_start_at[ u + 1 ]
                                                         : static_cast< node_t >( m );
  }
};


namespace gpu {

int         constexpr sparse_block      = 256;  // threads per block for two-hop
std::size_t constexpr max_bitmap_words  = 8192; // 32 KiB of smem per block

/*
 * ---------- CSR construction = counting sort on edge sources ----------
 * (Course Counting Sort / Radix Sort notes: histogram -> exclusive scan -> scatter.)
 */

/** Step 1: histogram of out-degrees. */
__global__
void count_degrees( edge_t const * edge_list, std::size_t m, node_t * degree )
{
    std::size_t const stride = gridDim.x * static_cast< std::size_t >( blockDim.x );
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
         i < m; i += stride )
    {
        atomicAdd( &degree[ edge_list[ i ].x ], 1 );
    }
}

/**
 * Atomic scatter is not stable, so sort each (short) row to get a canonical CSR,
 * e.g., [1, 2, 0, 0, 3, 2] for the handout example. One thread per row.
 */
__global__
void sort_rows( SparseGraph g )
{
    std::size_t const stride = gridDim.x * static_cast< std::size_t >( blockDim.x );
    for( std::size_t u = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
         u < g.n; u += stride )
    {
        node_t const begin = g.row_begin( static_cast< node_t >( u ) );
        node_t const end   = g.row_end  ( static_cast< node_t >( u ) );
        for( node_t i = begin + 1; i < end; ++i )
        {
            node_t const key = g.neighbours[ i ];
            node_t j = i - 1;
            while( j >= begin && g.neighbours[ j ] > key )
            {
                g.neighbours[ j + 1 ] = g.neighbours[ j ];
                --j;
            }
            g.neighbours[ j + 1 ] = key;
        }
    }
}

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 * This kernel is step 3 of the counting sort (scatter): each edge claims the next
 * free slot in its source's row. Steps 1-2 (histogram, scan) and the row sort are
 * launched by the host-side build_graph().
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */
__global__
void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t * cursor )
{
    std::size_t const stride = gridDim.x * static_cast< std::size_t >( blockDim.x );
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
         i < m; i += stride )
    {
        edge_t const e   = edge_list[ i ];
        node_t const pos = atomicAdd( &cursor[ e.x ], 1 );
        g.neighbours[ pos ] = e.y;
    }
}

/*
 * ---------- Two-hop reachability ----------
 *
 * One thread block per source node u. The block marks every w reachable as u -> v -> w
 * in a shared-memory bitmap (atomicOr), so duplicates collapse for free.
 *  - warps take different v in N(u); the 32 lanes of a warp read N(v) contiguously (coalesced)
 *  - w == u is skipped (walk u-v-u is not a path); v != u and v != w hold automatically
 *    because the graph has no self loops.
 * The bitmap is then read back word by word: __popc gives each thread its count, a block-level
 * exclusive scan (warp shuffles + smem) gives its write offset, and set bits are emitted in
 * increasing order, so each output row comes out sorted.
 *
 * If n exceeds the bitmap size, the column range [0,n) is processed in windows.
 *
 * Two passes (count, then fill) since the output size is unknown up front:
 *   WRITE == false : counts[u] = |two-hop(u)|
 *   WRITE == true  : out[ offsets[u] ... ] = two-hop(u)
 * Work per node is O(sum_{v in N(u)} deg(v) + n/32) versus Theta(n^2) per row for dense GEMM.
 */
template < bool WRITE >
__global__
void two_hop_pass( SparseGraph g, node_t * counts_or_offsets, node_t * out )
{
    extern __shared__ uint32_t bitmap[];
    __shared__ int warp_sums[ warp_size ];

    int const lane_id   = threadIdx.x % warp_size;
    int const warp_id   = threadIdx.x / warp_size;
    int const num_warps = blockDim.x  / warp_size;

    std::size_t const words_total  = ( g.n + 31 ) / 32;
    std::size_t const window_words = words_total < max_bitmap_words ? words_total : max_bitmap_words;

    for( std::size_t uu = blockIdx.x; uu < g.n; uu += gridDim.x ) // block-uniform loop
    {
        node_t const u     = static_cast< node_t >( uu );
        node_t const begin = g.row_begin( u );
        node_t const end   = g.row_end  ( u );

        int out_pos = WRITE ? counts_or_offsets[ u ] : 0;
        int count   = 0;

        for( std::size_t w0 = 0; w0 < words_total; w0 += window_words )
        {
            std::size_t const wwords = ( words_total - w0 < window_words ) ? words_total - w0 : window_words;
            node_t const lo = static_cast< node_t >( w0 * 32 );
            node_t const hi = static_cast< node_t >( ( w0 + wwords ) * 32 );

            // clear this window of the bitmap
            for( std::size_t i = threadIdx.x; i < wwords; i += blockDim.x ) { bitmap[ i ] = 0; }
            __syncthreads();

            // mark all w with u -> v -> w
            for( node_t i = begin + warp_id; i < end; i += num_warps )
            {
                node_t const v     = g.neighbours[ i ];
                node_t const v_end = g.row_end( v );
                for( node_t j = g.row_begin( v ) + lane_id; j < v_end; j += warp_size )
                {
                    node_t const w = g.neighbours[ j ];
                    if( w != u && w >= lo && w < hi )
                    {
                        node_t const bit = w - lo;
                        atomicOr( &bitmap[ bit >> 5 ], 1U << ( bit & 31 ) );
                    }
                }
            }
            __syncthreads();

            // compact set bits, blockDim.x words at a time
            for( std::size_t c = 0; c < wwords; c += blockDim.x )
            {
                std::size_t const idx  = c + threadIdx.x;
                uint32_t          word = ( idx < wwords ) ? bitmap[ idx ] : 0U;

                int total;
                int const prefix = block_exclusive_scan( __popc( word ), total, warp_sums );

                if( WRITE )
                {
                    int p = out_pos + prefix;
                    while( word )
                    {
                        int const b = __ffs( word ) - 1;
                        out[ p++ ] = lo + static_cast< node_t >( idx * 32 ) + b;
                        word &= word - 1;
                    }
                }
                out_pos += total;
                count   += total;
            }
            // block_exclusive_scan ends with __syncthreads(), so every bitmap read is done
            // before the next window (or next u) clears it
        }

        if( !WRITE && threadIdx.x == 0 ) { counts_or_offsets[ u ] = count; }
    }
}

} // namespace gpu


/**
 * Host-side driver for CSR construction (counting sort by source node).
 */
inline
void build_graph( SparseGraph & g, edge_t const * d_edges, std::size_t m )
{
    int constexpr threads = 256;
    std::size_t const edge_blocks = std::min< std::size_t >( ( m   + threads - 1 ) / threads, 65535 );
    std::size_t const node_blocks = std::min< std::size_t >( ( g.n + threads - 1 ) / threads, 65535 );

    node_t * d_scratch; // degree histogram, then reused as the scatter cursor
    CUDA_CALL( cudaMalloc( (void**)&d_scratch, sizeof( node_t ) * g.n ) );
    CUDA_CALL( cudaMemsetAsync( d_scratch, 0, sizeof( node_t ) * g.n ) );

    // 1. histogram
    if( edge_blocks > 0 ) { gpu::count_degrees<<< edge_blocks, threads >>>( d_edges, m, d_scratch ); }
    // 2. exclusive scan -> row offsets
    exclusive_scan( d_scratch, g.neighbours_start_at, g.n );
    // 3. scatter, using a copy of the offsets as per-row cursors
    CUDA_CALL( cudaMemcpyAsync( d_scratch, g.neighbours_start_at, sizeof( node_t ) * g.n, cudaMemcpyDeviceToDevice ) );
    if( edge_blocks > 0 ) { gpu::build_graph<<< edge_blocks, threads >>>( g, d_edges, m, d_scratch ); }
    // 4. canonical (sorted) rows
    if( node_blocks > 0 ) { gpu::sort_rows<<< node_blocks, threads >>>( g ); }
    CHECK_ERROR( "sparse build_graph" );

    CUDA_CALL( cudaFree( d_scratch ) );
}

/**
 * Host-side driver: replaces g's CSR arrays with its two-hop neighbourhood.
 * Note: the new graph can have more than m edges, so `neighbours` is reallocated.
 */
inline
void two_hop_reachability( SparseGraph & g )
{
    if( g.n == 0 ) { return; }

    std::size_t const words_total  = ( g.n + 31 ) / 32;
    std::size_t const window_words = std::min( words_total, gpu::max_bitmap_words );
    std::size_t const smem_bytes   = window_words * sizeof( uint32_t );
    unsigned    const blocks       = static_cast< unsigned >( std::min< std::size_t >( g.n, 65535 ) );

    node_t * d_counts, * d_offsets;
    CUDA_CALL( cudaMalloc( (void**)&d_counts,  sizeof( node_t ) * g.n ) );
    CUDA_CALL( cudaMalloc( (void**)&d_offsets, sizeof( node_t ) * g.n ) );

    // pass 1: count the size of each new row
    gpu::two_hop_pass< false ><<< blocks, gpu::sparse_block, smem_bytes >>>( g, d_counts, nullptr );
    CHECK_ERROR( "two_hop count pass" );

    // new row offsets and total edge count
    exclusive_scan( d_counts, d_offsets, g.n );
    node_t last_offset, last_count;
    CUDA_CALL( cudaMemcpy( &last_offset, d_offsets + g.n - 1, sizeof( node_t ), cudaMemcpyDeviceToHost ) );
    CUDA_CALL( cudaMemcpy( &last_count,  d_counts  + g.n - 1, sizeof( node_t ), cudaMemcpyDeviceToHost ) );
    std::size_t const new_m = static_cast< std::size_t >( last_offset ) + last_count;

    // pass 2: fill (reads the old graph, writes a fresh array)
    node_t * d_new_neighbours;
    CUDA_CALL( cudaMalloc( (void**)&d_new_neighbours, sizeof( node_t ) * std::max< std::size_t >( new_m, 1 ) ) );
    gpu::two_hop_pass< true ><<< blocks, gpu::sparse_block, smem_bytes >>>( g, d_offsets, d_new_neighbours );
    CHECK_ERROR( "two_hop fill pass" );

    // swap in the new graph
    CUDA_CALL( cudaMemcpyAsync( g.neighbours_start_at, d_offsets, sizeof( node_t ) * g.n, cudaMemcpyDeviceToDevice ) );
    CUDA_CALL( cudaFree( g.neighbours ) );
    g.neighbours = d_new_neighbours;
    g.m          = new_m;

    CUDA_CALL( cudaFree( d_counts ) );
    CUDA_CALL( cudaFree( d_offsets ) );
}

} // namespace a2
} // namespace csc485b

In [8]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs a CPU reference, and then runs and verifies the GPU code.
 */

#include <algorithm> // std::sort, std::equal
#include <chrono>    // for timing
#include <cstdio>    // std::printf
#include <iostream>  // std::cout, std::endl
#include <iterator>  // std::ostream_iterator
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "data_generator.h"
#include "data_types.h"

using namespace csc485b;

struct Timing { long long build_us; long long reach_us; };

/**
 * Runs timing tests on a CUDA graph implementation.
 * Consists of independently constructing the graph and then
 * modifying it to its two-hop neighbourhood.
 * The graph is taken by reference because two_hop_reachability may swap its buffers.
 */
template < typename DeviceGraph >
Timing run( DeviceGraph & g, a2::edge_t const * d_edges, std::size_t m, bool do_two_hop = true )
{
    cudaDeviceSynchronize();
    auto const build_start = std::chrono::high_resolution_clock::now();

    a2::build_graph( g, d_edges, m );

    cudaDeviceSynchronize();
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    if( do_two_hop ) { a2::two_hop_reachability( g ); }

    cudaDeviceSynchronize();
    auto const end = std::chrono::high_resolution_clock::now();

    CHECK_ERROR( "run" );
    return { std::chrono::duration_cast< std::chrono::microseconds >( reachability_start - build_start ).count(),
             std::chrono::duration_cast< std::chrono::microseconds >( end - reachability_start ).count() };
}

/** Allocates a dense graph, runs the GPU code on it, and copies the matrix back. */
std::vector< a2::node_t > run_dense( a2::edge_t const * d_edges, std::size_t n, std::size_t m,
                                     Timing & t, bool do_two_hop = true )
{
    a2::node_t * d_matrix;
    CUDA_CALL( cudaMalloc( (void**)&d_matrix, sizeof( a2::node_t ) * n * n ) );
    a2::DenseGraph d_dg{ n, d_matrix };

    t = run( d_dg, d_edges, m, do_two_hop );

    std::vector< a2::node_t > host_matrix( d_dg.matrix_size() );
    CUDA_CALL( cudaMemcpy( host_matrix.data(), d_dg.adjacencyMatrix,
                           sizeof( a2::node_t ) * d_dg.matrix_size(), cudaMemcpyDeviceToHost ) );
    CUDA_CALL( cudaFree( d_dg.adjacencyMatrix ) );
    return host_matrix;
}

struct HostCSR { std::vector< a2::node_t > offsets, neighbours; };

/** Allocates a sparse graph, runs the GPU code on it, and copies the CSR back. */
HostCSR run_sparse( a2::edge_t const * d_edges, std::size_t n, std::size_t m,
                    Timing & t, bool do_two_hop = true )
{
    a2::node_t * d_offsets, * d_neighbours;
    CUDA_CALL( cudaMalloc( (void**)&d_offsets,    sizeof( a2::node_t ) * n ) );
    CUDA_CALL( cudaMalloc( (void**)&d_neighbours, sizeof( a2::node_t ) * std::max< std::size_t >( m, 1 ) ) );
    a2::SparseGraph d_sg{ n, m, d_offsets, d_neighbours };

    t = run( d_sg, d_edges, m, do_two_hop );

    HostCSR out{ std::vector< a2::node_t >( n ), std::vector< a2::node_t >( d_sg.m ) };
    CUDA_CALL( cudaMemcpy( out.offsets.data(), d_sg.neighbours_start_at, sizeof( a2::node_t ) * n, cudaMemcpyDeviceToHost ) );
    if( d_sg.m > 0 )
    {
        CUDA_CALL( cudaMemcpy( out.neighbours.data(), d_sg.neighbours, sizeof( a2::node_t ) * d_sg.m, cudaMemcpyDeviceToHost ) );
    }
    CUDA_CALL( cudaFree( d_sg.neighbours ) );
    CUDA_CALL( cudaFree( d_sg.neighbours_start_at ) );
    return out;
}

/* ---------------- CPU reference ---------------- */

using AdjList = std::vector< std::vector< a2::node_t > >;

AdjList cpu_adjacency( a2::edge_list_t const & edges, std::size_t n )
{
    AdjList adj( n );
    for( auto const & e : edges ) { adj[ e.x ].push_back( e.y ); }
    for( auto & row : adj ) { std::sort( row.begin(), row.end() ); }
    return adj;
}

AdjList cpu_two_hop( AdjList const & adj )
{
    std::size_t const n = adj.size();
    AdjList out( n );
    std::vector< char > seen( n, 0 );
    for( std::size_t u = 0; u < n; ++u )
    {
        for( auto v : adj[ u ] )
            for( auto w : adj[ v ] )
                if( static_cast< std::size_t >( w ) != u && !seen[ w ] ) { seen[ w ] = 1; out[ u ].push_back( w ); }
        for( auto w : out[ u ] ) { seen[ w ] = 0; }
        std::sort( out[ u ].begin(), out[ u ].end() );
    }
    return out;
}

bool matches_dense( std::vector< a2::node_t > const & matrix, AdjList const & expected )
{
    std::size_t const n = expected.size();
    std::vector< a2::node_t > want( n * n, 0 );
    for( std::size_t u = 0; u < n; ++u )
        for( auto w : expected[ u ] ) { want[ u * n + w ] = 1; }
    return want == matrix;
}

bool matches_sparse( HostCSR const & csr, AdjList const & expected )
{
    std::size_t pos = 0;
    for( std::size_t u = 0; u < expected.size(); ++u )
    {
        if( csr.offsets[ u ] != static_cast< a2::node_t >( pos ) ) { return false; }
        for( auto w : expected[ u ] )
        {
            if( pos >= csr.neighbours.size() || csr.neighbours[ pos++ ] != w ) { return false; }
        }
    }
    return pos == csr.neighbours.size();
}

/* ---------------- helpers ---------------- */

a2::edge_t * upload( a2::edge_list_t const & edges )
{
    a2::edge_t * d_edges;
    CUDA_CALL( cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * std::max< std::size_t >( edges.size(), 1 ) ) );
    CUDA_CALL( cudaMemcpy( d_edges, edges.data(), sizeof( a2::edge_t ) * edges.size(), cudaMemcpyHostToDevice ) );
    return d_edges;
}

template < typename T >
void print( char const * label, std::vector< T > const & v )
{
    std::cout << label << ": [";
    std::copy( v.cbegin(), v.cend(), std::ostream_iterator< T >( std::cout, " " ) );
    std::cout << "]" << std::endl;
}

/** The worked example from the assignment handout. */
void handout_example()
{
    std::cout << "=== Handout example: edges (0,1), (0,2), (2,3) ===" << std::endl;
    a2::edge_list_t const edges{ make_int2( 0, 1 ), make_int2( 1, 0 ),
                                 make_int2( 0, 2 ), make_int2( 2, 0 ),
                                 make_int2( 2, 3 ), make_int2( 3, 2 ) };
    std::size_t const n = 4, m = edges.size();
    a2::edge_t * d_edges = upload( edges );
    Timing t;

    print( "dense build      ", run_dense ( d_edges, n, m, t, false ) );
    HostCSR built = run_sparse( d_edges, n, m, t, false );
    print( "csr build offsets", built.offsets );
    print( "csr build nbrs   ", built.neighbours );

    print( "dense two-hop    ", run_dense ( d_edges, n, m, t ) );
    HostCSR hop = run_sparse( d_edges, n, m, t );
    print( "csr 2hop offsets ", hop.offsets );
    print( "csr 2hop nbrs    ", hop.neighbours );

    CUDA_CALL( cudaFree( d_edges ) );
}

/** Runs dense and sparse on one random graph, verifies both, prints one table row. */
void benchmark( std::size_t n, std::size_t expected_degree )
{
    a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );
    std::size_t const m = graph.size();
    double const density = static_cast< double >( m ) / ( static_cast< double >( n ) * ( n - 1 ) );

    a2::edge_t * d_edges = upload( graph );
    Timing td, ts;
    auto const    dense  = run_dense ( d_edges, n, m, td );
    HostCSR const sparse = run_sparse( d_edges, n, m, ts );
    CUDA_CALL( cudaFree( d_edges ) );

    AdjList const expected = cpu_two_hop( cpu_adjacency( graph, n ) );
    bool const dense_ok  = matches_dense ( dense,  expected );
    bool const sparse_ok = matches_sparse( sparse, expected );

    std::printf( "%6zu %9zu %8.4f | %8lld %10lld %-4s | %8lld %10lld %-4s | %s\n",
                 n, m, density,
                 td.build_us, td.reach_us, dense_ok  ? "OK" : "FAIL",
                 ts.build_us, ts.reach_us, sparse_ok ? "OK" : "FAIL",
                 ts.reach_us < td.reach_us ? "sparse" : "dense" );
}

int main()
{
    // warm up the CUDA context so the first timing isn't polluted
    cudaFree( 0 );

    handout_example();

    // original starter demo: small random graph, echo input and dense output
    {
        std::cout << "\n=== Starter demo (n = 8) ===" << std::endl;
        std::size_t constexpr n = 8;
        std::size_t constexpr expected_degree = n >> 1;
        a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );
        for( auto const & e : graph ) { std::cout << "(" << e.x << "," << e.y << ") "; }
        std::cout << std::endl;

        a2::edge_t * d_edges = upload( graph );
        Timing t;
        print( "dense two-hop", run_dense( d_edges, n, graph.size(), t ) );
        HostCSR const csr = run_sparse( d_edges, n, graph.size(), t );
        print( "csr offsets  ", csr.offsets );
        print( "csr nbrs     ", csr.neighbours );
        CUDA_CALL( cudaFree( d_edges ) );
    }

    std::cout << "\n=== Benchmark (times in us; reachability compared) ===" << std::endl;
    std::printf( "%6s %9s %8s | %8s %10s %-4s | %8s %10s %-4s | %s\n",
                 "n", "m", "density", "d.build", "d.reach", "chk", "s.build", "s.reach", "chk", "faster" );

    // expected_degree d gives density ~ (floor(100 d / (n-1)) + 1) / 101 with this generator
    std::size_t const sizes[] = { 1024, 2048, 4096 };
    for( auto n : sizes )
    {
        benchmark( n, n / 200 );  // ~1%  (below the 2.5% threshold)
        benchmark( n, n / 80  );  // ~2%  (below the 2.5% threshold)
        benchmark( n, n / 20  );  // ~6%
        if( n <= 2048 ) { benchmark( n, n / 5 ); } // ~20% (CPU check gets slow beyond this)
    }

    return EXIT_SUCCESS;
}

In [9]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

=== Handout example: edges (0,1), (0,2), (2,3) ===
dense build      : [0 1 1 0 1 0 0 0 1 0 0 1 0 0 1 0 ]
csr build offsets: [0 2 3 5 ]
csr build nbrs   : [1 2 0 0 3 2 ]
dense two-hop    : [0 0 0 1 0 0 1 0 0 1 0 0 1 0 0 0 ]
csr 2hop offsets : [0 1 2 3 ]
csr 2hop nbrs    : [3 2 1 0 ]

=== Starter demo (n = 8) ===
(0,2) (2,0) (0,3) (3,0) (0,4) (4,0) (0,5) (5,0) (0,6) (6,0) (1,3) (3,1) (1,4) (4,1) (1,5) (5,1) (1,7) (7,1) (2,3) (3,2) (2,5) (5,2) (3,4) (4,3) (3,5) (5,3) (3,6) (6,3) (4,5) (5,4) (4,6) (6,4) (4,7) (7,4) (5,6) (6,5) (5,7) (7,5) (6,7) (7,6) 
dense two-hop: [0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 ]
csr offsets  : [0 7 14 21 28 35 42 49 ]
csr nbrs     : [1 2 3 4 5 6 7 0 2 3 4 5 6 7 0 1 3 4 5 6 7 0 1 2 4 5 6 7 0 1 2 3 5 6 7 0 1 2 3 4 6 7 0 1 2 3 4 5 7 0 1 2 3 4 5 6 ]

=== Benchmark (times in us; reachability compared) ===
     n         m  density |  d.build    d.reach chk  |  s.build    s.reach